# Exploratory Data Analysis and Feature Engineering

## Objective

Construct a small, interpretable set of cross-sectional equity features using
only information available at time t, and examine their behavior before fitting
predictive models.

The feature set will capture:

- short-term return / reversal,
- short- and medium-horizon momentum,
- realized volatility,
- price position relative to a moving average,
- abnormal trading activity,
- market beta.

Feature definitions are fixed before inspecting final test-period performance.

Any analysis involving the future-return target will use only the training
period (2010–2018) to avoid contaminating the validation and test sets.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

sys.path.append(str(SRC_DIR))

from config import UNIVERSE, MARKET_TICKER

In [2]:
df = pd.read_csv(
    DATA_DIR / "alpha_dataset_base.csv",
    parse_dates=[
        "date",
        "next_obs_date",
        "next_market_date",
    ],
)

df.head()

raw = pd.read_csv(DATA_DIR / "raw_prices.csv", parse_dates=["date"])

In [3]:
print(df.shape)
print(df["ticker"].nunique())
print(df["date"].min(), df["date"].max())
print(df["split"].value_counts())

(125610, 14)
30
2010-01-04 00:00:00 2026-08-26 00:00:00
split
train         67920
test          34980
validation    22710
Name: count, dtype: int64


In [4]:
RAW_FEATURES = [
    "ret_1d",
    "mom_5",
    "mom_20",
    "mom_60",
    "vol_20",
    "vol_60",
    "price_ma20_gap",
    "dollar_volume_z20",
    "beta_60",
]

## Momentum Features

We define $k$-day momentum as: $$ mom_{i,t}^{(k)} = \frac{P_{i,t}}{P_{i, t-k}} - 1 $$

In [5]:
def add_momentum_features(df):
    """
    Add trailing price-return features.

    mom_k(t) = P_t / P_{t-k} - 1
    """

    out = df.sort_values(["ticker", "date"]).copy()

    for k in [5, 20, 60]:
        out[f"mom_{k}"] = out["adj_close"] / out.groupby("ticker")["adj_close"].shift(k) - 1

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)

In [6]:
df = add_momentum_features(df)
df.head()

,date,adj_close,close,high,low,open,volume,ticker,ret_1d,next_obs_date,fwd_ret_1d,next_market_date,target_cs_1d,split,mom_5,mom_20,mom_60
0,2010-01-04,6.400960,7.643214,7.660714,7.585000,7.622500,493729600,AAPL,NaN,2010-01-05,0.001729,2010-01-05,-0.004657,train,NaN,NaN,NaN
1,2010-01-04,38.523415,57.720001,57.869999,56.560001,56.630001,5277400,AMGN,NaN,2010-01-05,-0.008663,2010-01-05,-0.015049,train,NaN,NaN,NaN
2,2010-01-04,43.777538,56.180000,56.389999,54.799999,55.720001,6186700,BA,NaN,2010-01-05,0.032752,2010-01-05,0.026366,train,NaN,NaN,NaN
3,2010-01-04,11.977122,15.690000,15.750000,15.120000,15.240000,180845200,BAC,NaN,2010-01-05,0.032505,2010-01-05,0.026119,train,NaN,NaN,NaN
4,2010-01-04,24.852760,34.000000,34.099998,33.400002,33.599998,40679350,C,NaN,2010-01-05,0.038235,2010-01-05,0.031849,train,NaN,NaN,NaN


mom_5 uses prices through today: $$ P_t, P_{t-1}, \ldots, P_{t-5} $$ and predicts $$ r_{t+1} $$ so there is no leakage

## Volatility Features

Use trailing standard deviation of daily returns: $$ vol_{i,t}^{(20)} ( r_{i, t-19}, \ldots r_{i,t}) $$

In [7]:
def add_volatility_features(df):
    """
    Add trailing realized-volatility features
    using daily simple returns.
    """

    out = df.sort_values(["ticker", "date"]).copy()

    for window in [20, 60]:
        out[f"vol_{window}"] = out.groupby("ticker")["ret_1d"].transform(
            lambda s: s.rolling(window=window, min_periods=window).std()
        )

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)

In [8]:
df = add_volatility_features(df)
df.head()

,date,adj_close,close,high,low,open,volume,ticker,ret_1d,next_obs_date,fwd_ret_1d,next_market_date,target_cs_1d,split,mom_5,mom_20,mom_60,vol_20,vol_60
0,2010-01-04,6.400960,7.643214,7.660714,7.585000,7.622500,493729600,AAPL,NaN,2010-01-05,0.001729,2010-01-05,-0.004657,train,NaN,NaN,NaN,NaN,NaN
1,2010-01-04,38.523415,57.720001,57.869999,56.560001,56.630001,5277400,AMGN,NaN,2010-01-05,-0.008663,2010-01-05,-0.015049,train,NaN,NaN,NaN,NaN,NaN
2,2010-01-04,43.777538,56.180000,56.389999,54.799999,55.720001,6186700,BA,NaN,2010-01-05,0.032752,2010-01-05,0.026366,train,NaN,NaN,NaN,NaN,NaN
3,2010-01-04,11.977122,15.690000,15.750000,15.120000,15.240000,180845200,BAC,NaN,2010-01-05,0.032505,2010-01-05,0.026119,train,NaN,NaN,NaN,NaN,NaN
4,2010-01-04,24.852760,34.000000,34.099998,33.400002,33.599998,40679350,C,NaN,2010-01-05,0.038235,2010-01-05,0.031849,train,NaN,NaN,NaN,NaN,NaN


## Price-position feature

We want to measure how far today's price is from its recent average:
$$ MA_{20,t} = \frac{1}{20} \sum_{j=0}^{19} P_{t-j} $$
and
$$ price_ma20_gap = \frac{P_t}{MA_{20,t}} - 1 $$

In [9]:
def add_price_features(df):
    """
    Add price-position feature relative to
    trailing 20-day moving average.
    """

    out = df.sort_values(["ticker", "date"]).copy()

    ma20 = out.groupby("ticker")["adj_close"].transform(lambda s: s.rolling(20, min_periods=20).mean())

    out["price_ma20_gap"] = out["adj_close"] / ma20 - 1

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)

In [10]:
df = add_price_features(df)
df.head()

,date,adj_close,close,high,low,open,volume,ticker,ret_1d,next_obs_date,fwd_ret_1d,next_market_date,target_cs_1d,split,mom_5,mom_20,mom_60,vol_20,vol_60,price_ma20_gap
0,2010-01-04,6.400960,7.643214,7.660714,7.585000,7.622500,493729600,AAPL,NaN,2010-01-05,0.001729,2010-01-05,-0.004657,train,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-04,38.523415,57.720001,57.869999,56.560001,56.630001,5277400,AMGN,NaN,2010-01-05,-0.008663,2010-01-05,-0.015049,train,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-04,43.777538,56.180000,56.389999,54.799999,55.720001,6186700,BA,NaN,2010-01-05,0.032752,2010-01-05,0.026366,train,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-04,11.977122,15.690000,15.750000,15.120000,15.240000,180845200,BAC,NaN,2010-01-05,0.032505,2010-01-05,0.026119,train,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-01-04,24.852760,34.000000,34.099998,33.400002,33.599998,40679350,C,NaN,2010-01-05,0.038235,2010-01-05,0.031849,train,NaN,NaN,NaN,NaN,NaN,NaN


## Trading-activity feature
Raw volume of stocks traded is not comparable across stocks so we calculate approximate dollar value

$$ DV_{i,t} = P_{i,t} \times Volume_{i,t} $$

and then take its log to see whether today's activity is unusual relative to that of stock's own previous 20 days

$$ z_{i,t}^{DV} = \frac{ \log(DV_{i,t}) - \mu_{i,t,20}^{DV}}{\sigma_{i,t,20}^{DV}} $$

In [11]:
def add_volume_features(df):
    """
    Measure abnormal dollar trading volume relative
    to each stock's own trailing 20-day history.
    """

    out = df.sort_values(["ticker", "date"]).copy()

    out["dollar_volume"] = (
        out["close"]
        * out["volume"]  # Dollar trading activity means approximately: shares traded today * actual market price today
    )

    out["log_dollar_volume"] = np.log(out["dollar_volume"])

    # Historical information only. At date t, this contains the value form t-1
    lagged_log_volume = out.groupby("ticker")["log_dollar_volume"].shift(1)

    rolling_mean = lagged_log_volume.groupby(out["ticker"]).transform(lambda s: s.rolling(20, min_periods=20).mean())

    rolling_std = lagged_log_volume.groupby(out["ticker"]).transform(lambda s: s.rolling(20, min_periods=20).std())

    out["dollar_volume_z20"] = (out["log_dollar_volume"] - rolling_mean) / rolling_std

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)


In [12]:
df = add_volume_features(df)

## Market beta

For a stock $i$, the market beta is defined as $$ \beta_i = \frac{Cov(r_i, r_M)}{Var(r_M)} $$
where $M$ represents the market (SPY)

e.g., $\beta = 1.5 $ means that the stock historically tends to move roughly 1.5 times as much as the market, in the same direction.

We'll calculate a rolling 60-day beta for our model

In [13]:
# First we calculate SPY return using the raw data

spy = raw[raw["ticker"] == MARKET_TICKER].sort_values("date").copy()

spy["spy_ret_1d"] = spy["adj_close"].pct_change(fill_method=None)

spy = spy[["date", "spy_ret_1d"]]

In [14]:
# Merge the SPY returns with the data
df = df.merge(spy, on="date", how="left")
df.head()

,date,adj_close,close,high,low,open,volume,ticker,ret_1d,next_obs_date,...,mom_5,mom_20,mom_60,vol_20,vol_60,price_ma20_gap,dollar_volume,log_dollar_volume,dollar_volume_z20,spy_ret_1d
0,2010-01-04,6.400960,7.643214,7.660714,7.585000,7.622500,493729600,AAPL,NaN,2010-01-05,...,NaN,NaN,NaN,NaN,NaN,NaN,3.773681e+09,22.051317,NaN,NaN
1,2010-01-04,38.523415,57.720001,57.869999,56.560001,56.630001,5277400,AMGN,NaN,2010-01-05,...,NaN,NaN,NaN,NaN,NaN,NaN,3.046115e+08,19.534548,NaN,NaN
2,2010-01-04,43.777538,56.180000,56.389999,54.799999,55.720001,6186700,BA,NaN,2010-01-05,...,NaN,NaN,NaN,NaN,NaN,NaN,3.475688e+08,19.666473,NaN,NaN
3,2010-01-04,11.977122,15.690000,15.750000,15.120000,15.240000,180845200,BAC,NaN,2010-01-05,...,NaN,NaN,NaN,NaN,NaN,NaN,2.837461e+09,21.766176,NaN,NaN
4,2010-01-04,24.852760,34.000000,34.099998,33.400002,33.599998,40679350,C,NaN,2010-01-05,...,NaN,NaN,NaN,NaN,NaN,NaN,1.383098e+09,21.047592,NaN,NaN


In [15]:
# Now we calculate the beta
def add_beta_feature(df, window=60):
    """
    Add rolling market beta:

        beta = Cov(stock return, SPY return)
               -----------------------------
                     Var(SPY return)
    """

    parts = []

    for ticker, d in df.groupby("ticker"):
        d = d.sort_values("date").copy()

        rolling_cov = d["ret_1d"].rolling(window, min_periods=window).cov(d["spy_ret_1d"])

        rolling_market_var = d["spy_ret_1d"].rolling(window, min_periods=window).var()

        d["beta_60"] = rolling_cov / rolling_market_var

        parts.append(d)

    out = pd.concat(parts, ignore_index=True)

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)

In [16]:
df = add_beta_feature(df)

In [17]:
# Let's inspect one stock before transforming anything
df[df["ticker"] == "AAPL"][
    [
        "date",
        "ret_1d",
        "mom_5",
        "mom_20",
        "mom_60",
        "vol_20",
        "vol_60",
        "price_ma20_gap",
        "dollar_volume_z20",
        "beta_60",
        "target_cs_1d",
    ]
].tail(10)

,date,ret_1d,mom_5,mom_20,mom_60,vol_20,vol_60,price_ma20_gap,dollar_volume_z20,beta_60,target_cs_1d
125310,2026-08-13,0.009959,-0.022044,-0.083228,0.025797,0.021437,0.020374,-0.044857,-0.998511,0.216134,0.003425
125340,2026-08-14,0.002195,-0.022775,-0.082538,0.024162,0.021448,0.020370,-0.038621,-2.015996,0.222817,0.004355
125370,2026-08-17,-0.001111,-0.008662,-0.063494,0.011922,0.021061,0.020326,-0.036553,-0.868281,0.205786,0.015277
125400,2026-08-18,0.014529,0.016792,-0.053221,0.017402,0.021383,0.020377,-0.019862,0.205624,0.177769,0.017667
125430,2026-08-19,0.021933,0.048238,-0.026962,0.026822,0.022049,0.020507,0.003028,0.078476,0.176339,-0.001608
125460,2026-08-20,-0.017454,0.019786,-0.031373,0.010503,0.022198,0.020638,-0.012904,-0.613064,0.212500,-0.014292
125490,2026-08-21,-0.006264,0.011179,-0.070276,-0.003967,0.020462,0.020630,-0.015437,-0.235342,0.207547,0.004381
125520,2026-08-24,0.003200,0.015544,-0.078069,-0.006087,0.020219,0.020623,-0.008138,-1.083551,0.200160,-0.003153
125550,2026-08-25,-0.001418,-0.000419,-0.087958,-0.006065,0.019989,0.020623,-0.004791,-1.819452,0.199778,0.012997
125580,2026-08-26,0.011455,-0.010668,-0.072355,0.024192,0.020296,0.020528,0.010576,-0.844729,0.210042,0.006576


## Cross-sectional normalization

If on a given day the 20-day momentum for AAPL, MSFT, JPM, and XOM are 15%, 10%, 4%, and -2% respectively, then instead of giving the regression model the absolute numbers, we provide the normalized values.

The question we care about is: Which stocks have unusually high momentum relative to the other stocks today?

So for every date, we calculate $$ z_{i,t} = \frac{ x_{i,t} - {\bar{x}_t}} {s_t} $$ where $ \bar{x}_t$ and $ s_t$ are cross-section momentum mean and standard deviation.

If any stock's momentum is greater than that day's universe average, it gets a positive z-score. 

This makes the project explicitly cross-sectional.

In [18]:
def cross_sectional_standardize(df, feature_cols, clip=None):
    """
    Standardize each feature across stocks independently
    on each date.

        z = (x - cross-sectional mean)
            / cross-sectional std

    Optionally clip extreme z-scores.
    """

    out = df.copy()

    for feature in feature_cols:
        cs_mean = out.groupby("date")[feature].transform("mean")

        cs_std = out.groupby("date")[feature].transform("std")

        z = (out[feature] - cs_mean) / cs_std

        if clip is not None:
            z = z.clip(-clip, clip)

        out[f"{feature}_z"] = z

    return out

In [19]:
df = cross_sectional_standardize(df, RAW_FEATURES, clip=None)

In [20]:
# The actual model inputs will be
MODEL_FEATURES = [
    f"{feature}_z"  # _z represents cross-sectional standardization across stocks on that day
    for feature in RAW_FEATURES
]

In [21]:
MODEL_FEATURES

['ret_1d_z',
 'mom_5_z',
 'mom_20_z',
 'mom_60_z',
 'vol_20_z',
 'vol_60_z',
 'price_ma20_gap_z',
 'dollar_volume_z20_z',
 'beta_60_z']

In [22]:
# Verifying the cross-sectional standardization
check_date = df.dropna(subset=MODEL_FEATURES)["date"].iloc[0]

check = df[df["date"] == check_date]

check[MODEL_FEATURES].agg(["mean", "std"])

,ret_1d_z,mom_5_z,mom_20_z,mom_60_z,vol_20_z,vol_60_z,price_ma20_gap_z,dollar_volume_z20_z,beta_60_z
mean,-1.480297e-17,-1.850372e-18,-1.369275e-16,3.330669e-17,7.771561e-17,-4.810966e-17,1.850372e-17,3.053113e-17,-1.646831e-16
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


## Missingness

In [23]:
missing = df[RAW_FEATURES + MODEL_FEATURES].isna().mean().sort_values(ascending=False)

missing

vol_60                 0.014330
mom_60                 0.014330
vol_60_z               0.014330
beta_60_z              0.014330
beta_60                0.014330
mom_60_z               0.014330
dollar_volume_z20_z    0.004777
vol_20_z               0.004777
vol_20                 0.004777
mom_20                 0.004777
dollar_volume_z20      0.004777
mom_20_z               0.004777
price_ma20_gap_z       0.004538
price_ma20_gap         0.004538
mom_5_z                0.001194
mom_5                  0.001194
ret_1d                 0.000239
ret_1d_z               0.000239
dtype: float64

The missingness values make sense as it is just the required rolling-window warm-up.

* vol_60, mom_60, beta_60 : 60/4188 ~ 1.433%
* mom_20, vol_20: 20/4188 ~ 0.4777%
* mom_5: 5/4188 ~ 0.1194%
* ret_1d: 1/4188 ~ 0.0239%

All of this imply that the missingness is simply because of needing enough values to calculate quantities and not because of having random missing data, e.g., we cannot calculate mom_60 until we have 60 previous observations.

In [24]:
# Dropping the Nan/missing values to get a complete feature set charts
complete_features = df.dropna(subset=MODEL_FEATURES)

complete_features["date"].min()


Timestamp('2010-03-31 00:00:00')

In [25]:
complete_features.groupby("split")["date"].agg(["min", "max", "nunique"]).reindex(["train", "validation", "test"])

,min,max,nunique
split,,,
train,2010-03-31,2018-12-31,2204
validation,2019-01-02,2021-12-31,757
test,2022-01-03,2026-08-26,1166


## Creating modeling dataset

We require:
* all nine features
* valid target

In [26]:
model_df = df.dropna(subset=MODEL_FEATURES + ["target_cs_1d", "fwd_ret_1d"]).copy()

In [27]:
print(model_df.shape)

print(model_df.groupby("split")["date"].agg(["min", "max", "nunique"]).reindex(["train", "validation", "test"]))

(123810, 34)
                  min        max  nunique
split                                    
train      2010-03-31 2018-12-31     2204
validation 2019-01-02 2021-12-31      757
test       2022-01-03 2026-08-26     1166


## Basic feature distribution


In [28]:
train = model_df[model_df["split"] == "train"].copy()

In [29]:
# Describe raw features
train[RAW_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
ret_1d,66120.0,0.000537,0.014600,-0.203182,-0.006345,0.000475,0.007586,0.169983
mom_5,66120.0,0.002612,0.031597,-0.336392,-0.013725,0.003253,0.019772,0.306854
mom_20,66120.0,0.010783,0.059920,-0.371015,-0.023336,0.011818,0.045973,0.433619
mom_60,66120.0,0.034188,0.103074,-0.482540,-0.028710,0.033650,0.096740,0.966349
vol_20,66120.0,0.013110,0.006434,0.002383,0.008765,0.011722,0.015812,0.087268
vol_60,66120.0,0.013442,0.005477,0.003848,0.009769,0.012424,0.015948,0.063425
price_ma20_gap,66120.0,0.004375,0.033664,-0.320175,-0.013600,0.006237,0.024111,0.245193
dollar_volume_z20,66120.0,0.017650,1.231395,-8.643833,-0.749375,-0.105903,0.634275,15.038154
beta_60,66120.0,0.997632,0.416529,-0.615038,0.712578,0.964690,1.233725,3.338942


In [30]:
# Standardized features
train[MODEL_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
ret_1d_z,66120.0,9.402978e-19,0.9832,-5.074157,-0.578671,-0.011411,0.564742,5.135722
mom_5_z,66120.0,-1.343283e-20,0.9832,-4.612336,-0.596322,-0.011378,0.585107,5.031313
mom_20_z,66120.0,1.665670e-18,0.9832,-4.074370,-0.631073,-0.002440,0.628723,4.339098
mom_60_z,66120.0,8.966411e-19,0.9832,-3.579581,-0.676262,-0.030530,0.655527,3.921136
vol_20_z,66120.0,-5.970891e-18,0.9832,-2.568800,-0.714781,-0.149008,0.528934,4.604227
vol_60_z,66120.0,-6.917905e-18,0.9832,-2.431168,-0.719821,-0.104967,0.598159,4.073676
price_ma20_gap_z,66120.0,-7.656711e-19,0.9832,-4.365510,-0.617211,0.013822,0.622187,4.296005
dollar_volume_z20_z,66120.0,-1.386939e-18,0.9832,-4.514657,-0.656377,-0.124405,0.524837,5.070734
beta_60_z,66120.0,4.150743e-18,0.9832,-2.788531,-0.736087,-0.071596,0.680019,3.201721


## Feature correlation matrix

We expect some obvious relationships, particularly
$$ mom_{20} \leftrightarrow price\_ma20\_gap $$
and probably $$ mom_5 \leftrightarrow mom_{20} $$ and $$ vol_{20} \leftrightarrow vol_{60} $$

This is exactly where multicollinearity enters the project

In [31]:
corr = train[MODEL_FEATURES].corr()

fig = px.imshow(corr, text_auto=".2f", aspect="auto", title="Training-Period Feature Correlation Matrix")

fig.show()

From the correlation results, we see that there are mainly two correlated blocks.

#### Price/Momentum Block: 
ret_1d, mom_5, mom_20, mom_60, and price_ma20_gap are correlated which makes economic sense as they are all derived from recent price movement.

In particular, correlation between mom_20 and price_ma20_gap is very high (= 0.85), as they are both asking the similar questions. mom_20 looks at how much did the stock price rise in over ~1 month while price_ma20_gap looks at how far today's price is above its recent average.

#### Risk Block:
vol_20, vol_60, beta_60 are correlated as high-beta stocks tend to have larger market-driven fluctuations leading to a positive beta-volatility relationship.

Also, we notice that the activity is nearly independent as dollar_volume_z20 has correlations close to zero with almost everything else. This provides a different dimension of information.


## Information coefficient

We simply ask whether one feature by itself has any association with tomorrow's relative return

For a given date: $$ IC_t = Corr_i ( x_{i,t}, y_{i,t+1}) $$ 
e.g., $$ Corr_i ( {{mom_{20}}_{i,t}}, target_{i, t+1}) $$

If $$ IC_t > 0 $$ stocks with high momentum that day tended to others the next day and vice-versa

In [32]:
# Function for daily information coefficient
def daily_ic(df, feature, target="target_cs_1d", method="spearman"):
    """
    Compute cross-sectional correlation between one
    feature and future returns separately for each date.

    We focus on Spearman correlation, or Rank IC because eventually we will rank stock
    """

    def corr_one_day(d):

        x = d[feature]
        y = d[target]

        valid = x.notna() & y.notna()

        if valid.sum() < 5:
            return np.nan

        return x[valid].corr(y[valid], method=method)

    return df.groupby("date")[[feature, target]].apply(corr_one_day)

In [33]:
# Compute individual-feature ICs for the training period
ic_results = []

for feature in MODEL_FEATURES:
    pearson_ic = daily_ic(train, feature, method="pearson")

    rank_ic = daily_ic(train, feature, method="spearman")

    ic_results.append(
        {
            "feature": feature,
            "mean_pearson_ic": pearson_ic.mean(),
            "mean_rank_ic": rank_ic.mean(),
            "rank_ic_std": rank_ic.std(),
            "pct_rank_ic_positive": (rank_ic > 0).mean(),
        }
    )

ic_table = pd.DataFrame(ic_results).sort_values("mean_rank_ic", ascending=False)

ic_table

,feature,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
3,mom_60_z,0.002295,0.002574,0.282162,0.522686
7,dollar_volume_z20_z,-0.002161,0.002313,0.216536,0.495009
2,mom_20_z,-0.000982,-0.002498,0.274156,0.511343
5,vol_60_z,-0.000184,-0.005725,0.335313,0.495463
4,vol_20_z,-0.002915,-0.006926,0.321217,0.485027
8,beta_60_z,-0.005592,-0.010054,0.363017,0.489564
6,price_ma20_gap_z,-0.007581,-0.010130,0.273721,0.483212
1,mom_5_z,-0.010181,-0.015122,0.270476,0.480036
0,ret_1d_z,-0.009703,-0.017848,0.269269,0.478675


Even a small value of IC can represent a measurable relationship.

Also, we don't drop a feature just because it IC is weak as a variable can have weak standalone predictive power but improve predictions conditional on other features. Individual correlation is not the same as multivariate predictive value.

From the mean_rank_ic values above, we notice that short-horizon return features stand out the most suggesting a possible pattern of short-horizon reversal rather than short-horizon momentum. 
For example $$ IC( ret_{1d}, r_{t+1}) < 0 $$ means that, on average during the training sample, stocks that performed relatively well today tended to perform slighlty worse relative to peers tomorrow.
Similarly, $$ IC( mom_5, r_{t+1}) < 0 $$ represents a classic mean-reversion type result.

Note: There is an enormous amount of noise relative to the mean, e.g., for ret_1d we have mean = -0.0178 and std = 0.2693 meaning that IC can change a lot between one day and the next and only after averaging over thousands of dates does a small negative tendency appear.

In [34]:
# Save the finished dataset

model_df.to_csv(DATA_DIR / "alpha_dataset.csv", index=False)

Notebook 01 (01_data_pipeline) gives: $ \hspace{1.5cm} y_{i, t+1} \hspace{1.5cm} $ i.e., what happened tomorrow

Notebook 02 (02_eda_and_features) gives: $\hspace{1cm} X_{i,t} \hspace{1.7cm} $ i.e., what we knew today

Notebook 03 (next notebook) will ask: $ \hspace{1.5cm} \hat{y} _{i, t+1} = f(X_{i,t}) $

We will start with OLS and then move to Ridge, Elastic Net and XGBoost